# 03 Technical Indicators

## Purpose
Compute price- and volume-based technical indicators from Nikkei 225 OHLCV data,
then transform them into a **model-ready feature matrix** with no look-ahead bias.
This notebook produces the primary feature set for `04_modeling.ipynb`.

## Motivation per indicator group
| Group | Indicators | What they capture |
|-------|------------|-------------------|
| Moving averages | SMA 5/10/20/50/75/200, EMA 5/20/50 | Trend direction and momentum |
| Bollinger Bands | BB upper/lower/mid, %B, width | Volatility and mean-reversion zones |
| RSI | RSI 14, RSI 9 | Overbought / oversold momentum |
| MACD | MACD line, signal, histogram | Trend changes and crossover signals |
| Volume | Volume/SMA20 ratio, OBV | Conviction behind price moves |
| ATR | ATR 14, ATR % | Daily volatility regime |
| Stochastic | %K, %D | Short-term momentum extremes |

## Feature engineering principles
- **No look-ahead bias**: all features are derived from time-*t* data;
  target (`target_return`) is the time *t+1* log return.
- **Normalised features**: raw price levels are converted to price-relative
  forms (e.g. `Close / SMA_20 - 1`) so the model is scale-invariant.
- **Bounded indicators** (RSI, Stoch) are used as-is since they are already 0–100.

## Expected outputs
| Output | Location |
|--------|----------|
| `technical_features.csv` — 30+ feature columns + target | `data/processed/` |
| Indicator charts (MA, BB, RSI, MACD, Volume, ATR, Stoch) | `output/figures/` |
| Feature–target correlation bar chart | `output/figures/03_feature_corr.png` |

## Key things to look for
- Which features have the highest absolute correlation with `target_return`?
- Is `BB_pct` near 0 or 1? (mean-reversion signal)
- Does OBV trend diverge from price? (volume-price divergence)

## Sections
| # | Content |
|---|---------|
| 0 | Colab environment setup |
| 1 | Data loading + indicator computation |
| 2 | Moving averages |
| 3 | Bollinger Bands |
| 4 | RSI |
| 5 | MACD |
| 6 | Volume indicators |
| 7 | ATR |
| 8 | Stochastic oscillator |
| 9 | Feature engineering |
| 10 | Save to data/processed/ |


In [ ]:
# ── Step 1: Install packages ───────────────────────────────────────────────
import subprocess, sys

pkgs = ['yfinance', 'pandas-ta', 'scikit-learn>=1.5.0', 'statsmodels>=0.14.0']
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
    capture_output=True, text=True
)
out = result.stdout + result.stderr
print(out.strip() if out.strip() else 'All packages already up to date.')

if 'Successfully installed' in out:
    print('\nPackages upgraded — restarting runtime...')
    import os; os.kill(os.getpid(), 9)

In [ ]:
# ── Step 2: Repository & path setup ───────────────────────────────────────
import os, sys

REPO = '/content/Nikkei_Analysis'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(REPO):
        !git clone https://github.com/Takumi-Itokawa-Finance/Nikkei_Analysis.git {REPO}
    else:
        !git -C {REPO} pull
    os.chdir(REPO)

    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = '/content/drive/MyDrive/Nikkei_Analysis/data'
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if not os.path.exists(f'{REPO}/data'):
        os.symlink(DRIVE_DATA, f'{REPO}/data')

    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    print(f'Setup complete.  CWD={os.getcwd()}')
else:
    print('Running in local environment.')

## 1. Data Loading

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.data.fetcher import fetch_nikkei
from src.features.technical import compute_indicators, build_features

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

OUTPUT_FIGS = "output/figures"
OUTPUT_DATA = "data/processed"
os.makedirs(OUTPUT_FIGS, exist_ok=True)
os.makedirs(OUTPUT_DATA, exist_ok=True)


In [ ]:
raw = fetch_nikkei(period="5y", interval="1d")

# Flatten MultiIndex columns if present (yfinance >= 0.2)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

raw = raw[["Open", "High", "Low", "Close", "Volume"]].copy()
raw = raw.ffill().dropna()

# Compute all technical indicators via src module
df = compute_indicators(raw)

print(f"Shape  : {df.shape}")
print(f"Period : {df.index[0].date()} to {df.index[-1].date()}")
display(df[["Open", "High", "Low", "Close", "Volume"]].tail())


## 2. Moving Averages (SMA / EMA)

In [ ]:
sma_periods = [5, 10, 20, 50, 75, 200]
ema_periods = [5, 20, 50]
print("Moving averages computed.")
display(df[[f"SMA_{p}" for p in sma_periods] + [f"EMA_{p}" for p in ema_periods]].tail())


In [ ]:
plot_df = df.loc[df.index >= df.index[-1] - pd.DateOffset(years=1)]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(plot_df.index, plot_df['Close'], label='Close', color='black', linewidth=1.2)
for p, color in zip([20, 50, 200], ['steelblue', 'orange', 'red']):
    ax.plot(plot_df.index, plot_df[f'SMA_{p}'], label=f'SMA {p}', linewidth=1, color=color)
ax.set_title('Nikkei 225 — Close & Moving Averages (1Y)')
ax.set_ylabel('Price (JPY)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_moving_averages.png', dpi=150)
plt.show()

## 3. Bollinger Bands

In [ ]:
display(df[["Close", "BB_lower", "BB_mid", "BB_upper", "BB_width", "BB_pct"]].tail())


In [ ]:
plot_df = df.loc[df.index >= df.index[-1] - pd.DateOffset(years=1)]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2, label='Close')
axes[0].plot(plot_df.index, plot_df['BB_upper'], color='steelblue', linewidth=0.8, label='Upper')
axes[0].plot(plot_df.index, plot_df['BB_mid'],   color='grey',      linewidth=0.8, linestyle='--', label='Mid')
axes[0].plot(plot_df.index, plot_df['BB_lower'], color='steelblue', linewidth=0.8, label='Lower')
axes[0].fill_between(plot_df.index, plot_df['BB_lower'], plot_df['BB_upper'], alpha=0.1, color='steelblue')
axes[0].set_title('Bollinger Bands (20, 2)')
axes[0].set_ylabel('Price (JPY)')
axes[0].legend()

axes[1].plot(plot_df.index, plot_df['BB_pct'], color='purple', linewidth=1)
axes[1].axhline(1.0, color='steelblue', linewidth=0.8, linestyle='--', label='Overbought')
axes[1].axhline(0.0, color='tomato',    linewidth=0.8, linestyle='--', label='Oversold')
axes[1].set_title('%B (Bollinger Band Position)')
axes[1].set_ylabel('%B')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_bollinger_bands.png', dpi=150)
plt.show()

## 4. RSI (Relative Strength Index)

In [ ]:
display(df[["Close", "RSI_14", "RSI_9"]].tail())


In [ ]:
plot_df = df.loc[df.index >= df.index[-1] - pd.DateOffset(years=1)]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2)
axes[0].set_title('Nikkei 225 Close')
axes[0].set_ylabel('Price (JPY)')

axes[1].plot(plot_df.index, plot_df['RSI_14'], color='steelblue', linewidth=1, label='RSI 14')
axes[1].plot(plot_df.index, plot_df['RSI_9'],  color='orange',    linewidth=1, label='RSI 9', alpha=0.7)
axes[1].axhline(70, color='tomato',    linewidth=0.8, linestyle='--', label='Overbought (70)')
axes[1].axhline(30, color='steelblue', linewidth=0.8, linestyle='--', label='Oversold (30)')
axes[1].axhline(50, color='grey',      linewidth=0.6, linestyle=':')
axes[1].fill_between(plot_df.index, plot_df['RSI_14'], 70,
                     where=(plot_df['RSI_14'] >= 70), alpha=0.2, color='tomato')
axes[1].fill_between(plot_df.index, plot_df['RSI_14'], 30,
                     where=(plot_df['RSI_14'] <= 30), alpha=0.2, color='steelblue')
axes[1].set_title('RSI')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_rsi.png', dpi=150)
plt.show()

## 5. MACD

In [ ]:
display(df[["Close", "MACD", "MACD_signal", "MACD_hist"]].tail())


In [ ]:
plot_df = df.loc[df.index >= df.index[-1] - pd.DateOffset(years=1)]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2)
axes[0].set_title('Nikkei 225 Close')
axes[0].set_ylabel('Price (JPY)')

axes[1].plot(plot_df.index, plot_df['MACD'],        color='steelblue', linewidth=1,   label='MACD')
axes[1].plot(plot_df.index, plot_df['MACD_signal'], color='orange',    linewidth=1,   label='Signal')
axes[1].bar(plot_df.index,  plot_df['MACD_hist'],
            color=['steelblue' if v >= 0 else 'tomato' for v in plot_df['MACD_hist']],
            alpha=0.5, label='Histogram')
axes[1].axhline(0, color='black', linewidth=0.6)
axes[1].set_title('MACD (12, 26, 9)')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_macd.png', dpi=150)
plt.show()

## 6. Volume Indicators

In [ ]:
display(df[["Close", "Volume", "Volume_SMA20", "Volume_ratio", "OBV"]].tail())


In [ ]:
plot_df = df.loc[df.index >= df.index[-1] - pd.DateOffset(years=1)]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2)
axes[0].set_title('Nikkei 225 Close')
axes[0].set_ylabel('Price (JPY)')

axes[1].bar(plot_df.index, plot_df['Volume'], color='steelblue', alpha=0.6, label='Volume')
axes[1].plot(plot_df.index, plot_df['Volume_SMA20'], color='orange', linewidth=1, label='SMA 20')
axes[1].set_title('Volume')
axes[1].set_ylabel('Volume')
axes[1].legend()

axes[2].plot(plot_df.index, plot_df['OBV'], color='purple', linewidth=1)
axes[2].set_title('On-Balance Volume (OBV)')
axes[2].set_ylabel('OBV')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_volume.png', dpi=150)
plt.show()

## 7. ATR — Average True Range (Volatility)

In [ ]:
display(df[["Close", "ATR_14", "ATR_pct"]].tail())


In [ ]:
plot_df = df.loc[df.index >= df.index[-1] - pd.DateOffset(years=1)]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2)
axes[0].set_title('Nikkei 225 Close')
axes[0].set_ylabel('Price (JPY)')

axes[1].plot(plot_df.index, plot_df['ATR_pct'], color='darkorange', linewidth=1)
axes[1].fill_between(plot_df.index, plot_df['ATR_pct'], alpha=0.2, color='orange')
axes[1].set_title('ATR % (14-day, normalised by Close)')
axes[1].set_ylabel('ATR %')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_atr.png', dpi=150)
plt.show()

## 8. Stochastic Oscillator

In [ ]:
display(df[["Close", "Stoch_K", "Stoch_D"]].tail())


In [ ]:
plot_df = df.loc[df.index >= df.index[-1] - pd.DateOffset(years=1)]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2)
axes[0].set_title('Nikkei 225 Close')
axes[0].set_ylabel('Price (JPY)')

axes[1].plot(plot_df.index, plot_df['Stoch_K'], color='steelblue', linewidth=1, label='%K')
axes[1].plot(plot_df.index, plot_df['Stoch_D'], color='orange',    linewidth=1, label='%D')
axes[1].axhline(80, color='tomato',    linewidth=0.8, linestyle='--', label='Overbought (80)')
axes[1].axhline(20, color='steelblue', linewidth=0.8, linestyle='--', label='Oversold (20)')
axes[1].set_title('Stochastic Oscillator (14, 3)')
axes[1].set_ylabel('%K / %D')
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_stochastic.png', dpi=150)
plt.show()

## 9. Feature Engineering — Model-Ready Matrix

Convert raw indicators into features suitable for prediction.

Key principle: **features must be available at time t to predict t+1**.
All features are derived from same-day OHLCV — no look-ahead bias.

Raw price levels → normalised / stationary alternatives:

| Raw | Model-ready |
|-----|-------------|
| SMA_20 | Close / SMA_20 - 1 (price relative to MA) |
| BB_upper / lower | BB_pct (already 0-1 scaled) |
| Volume | Volume_ratio (vs 20-day average) |
| ATR | ATR_pct (% of price) |
| MACD | MACD_hist (already a difference) |
| RSI | as-is (bounded 0-100) |
| Stoch | as-is (bounded 0-100) |

In [ ]:
feat = build_features(df)

print(f"Feature matrix shape : {feat.shape}")
print(f"Period               : {feat.index[0].date()} to {feat.index[-1].date()}")
print(f"
Features ({len(feat.columns) - 2}):")
print([c for c in feat.columns if c not in ["target_return", "target_close"]])


In [ ]:
display(feat.describe().T)

In [ ]:
# Correlation of each technical feature with next-day return
feature_cols = [c for c in feat.columns if c not in ['target_return', 'target_close']]
corr_target = feat[feature_cols].corrwith(feat['target_return']).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['steelblue' if v >= 0 else 'tomato' for v in corr_target]
corr_target.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Technical Feature Correlation with Next-Day Return')
ax.set_xlabel('Pearson r')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_feature_corr.png', dpi=150)
plt.show()

print('\nTop 10 by absolute correlation:')
display(corr_target.abs().sort_values(ascending=False).head(10).to_frame('abs_corr'))

## 10. Save to data/processed/

In [ ]:
save_path = f'{OUTPUT_DATA}/technical_features.csv'
feat.to_csv(save_path)
print(f'Saved: {save_path}  shape={feat.shape}')
print('\nNext: 04_modeling.ipynb — load this file and train prediction models.')